# Biohub window-3 probe — unit-sliced training (STRATEGY 49)

**One-time setup:** add a Colab Secret named `KAGGLE_JSON` containing the full contents of your `kaggle.json` (Kaggle → Settings → API → Create New Token). Enable notebook access for the secret.

**Every session:** Runtime → Change runtime type → **A100** (L4 works, ~40% slower per unit). Then *Run all*. The session downloads data (~25 min), resumes from the last checkpoint, trains until the time budget, and uploads the checkpoint back to Kaggle. Re-run whenever you have compute units — progress accumulates automatically.

In [ ]:
#@title Session parameters
MAX_MINUTES = 165      #@param {type:"integer"}  — leave ~30 min of session for data+upload
BATCH_SIZE  = 8        #@param {type:"integer"}  — drop to 4 if OOM (L4/T4)
EPOCHS      = 60       #@param {type:"integer"}  — total target; sessions accumulate toward it
import torch, subprocess
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
print("GPU:", gpu)
assert gpu != "NONE", "Set Runtime -> A100/L4 first"


In [ ]:
#@title Dependencies (~2 min)
%pip -q install "zarr>=3.0.10,<4" blosc2 "numcodecs>=0.13,<0.16" "geff>=1.1.3.1.1" "geff-spec<1.2" tracksdata "polars>=1.36" dask tqdm rustworkx "sqlalchemy>=2" 2>&1 | tail -1
import zarr; print("zarr", zarr.__version__)


In [ ]:
#@title Kaggle auth (Secret KAGGLE_JSON: kaggle.json content OR a KGAT_ token)
import json, os, pathlib
from google.colab import userdata

try:
    raw = userdata.get('KAGGLE_JSON').strip()
except Exception as e:
    raise SystemExit(
        "Secret KAGGLE_JSON not readable. Left sidebar -> key icon -> Add secret\n"
        "named KAGGLE_JSON with 'Notebook access' ON, then re-run.") from e

# The old CLI preinstalled on Colab rejects KGAT tokens -- upgrade FIRST.
!pip -q install -U kaggle 2>&1 | tail -1

kdir = pathlib.Path('/root/.kaggle'); kdir.mkdir(exist_ok=True)
if raw.startswith('KGAT_'):
    (kdir / 'access_token').write_text(raw)
    os.chmod(kdir / 'access_token', 0o600)
    os.environ['KAGGLE_API_TOKEN'] = raw
    print('using KGAT access token')
else:
    try:
        creds = json.loads(raw)
        assert 'username' in creds and 'key' in creds
    except Exception:
        raise SystemExit(
            'KAGGLE_JSON must be either a KGAT_... token or the ENTIRE kaggle.json '
            'content like {"username":"x","key":"y"}. '
            f'Got {len(raw)} chars starting with: {raw[:12]!r}')
    os.environ['KAGGLE_USERNAME'] = creds['username']
    os.environ['KAGGLE_KEY'] = creds['key']
    for d in ('/root/.kaggle', '/root/.config/kaggle'):
        pathlib.Path(d).mkdir(parents=True, exist_ok=True)
        f = pathlib.Path(d) / 'kaggle.json'
        f.write_text(json.dumps(creds)); os.chmod(f, 0o600)
    print('using legacy kaggle.json credentials')

r = os.popen('kaggle competitions list -s biohub 2>&1').read()
print(r[:300])
assert 'biohub' in r.lower() and 'authentication' not in r.lower(), (
    'CLI still not authenticated. Check the token is active on kaggle.com/settings '
    'and that this account accepted the competition rules (competition page -> Join).')
print('AUTH OK')


In [ ]:
#@title Competition data (~25 min first run; cached on the session disk)
import os
if not os.path.exists('/content/data/train'):
    !mkdir -p /content/data
    !kaggle competitions download -c biohub-cell-tracking-during-development -p /content/data
    print("unzipping (this is the slow part)...")
    !cd /content/data && unzip -qo biohub-cell-tracking-during-development.zip && rm biohub-cell-tracking-during-development.zip
n = len([d for d in os.listdir('/content/data/train') if d.endswith('.zarr')])
print(f"train mount: {n} zarr stores")
assert n >= 190


In [ ]:
#@title Trainer: support pack + resume/budget/patience patch + splits
import base64, os, json, pathlib
if not os.path.exists('/content/repo'):
    !kaggle datasets download pilkwang/biohub-tracking-support-pack-50ep-v1 -p /content/sp --unzip
    !cp -r /content/sp/repo /content/repo

pathlib.Path('/content/patcher.py').write_bytes(base64.b64decode("IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJQYXRjaCB0aGUgZnVsbC1yZWNpcGUgdHJhaW5lciBmb3IgdW5pdC1zbGljZWQgQ29sYWIgdHJhaW5pbmcgKFNUUkFURUdZIDQ4LzQ5KS4KClRoZSBzdXBwb3J0LXBhY2sgYHRyYWluX3VuZXRfdHJhbnNmb3JtZXIucHlgIHNhdmVzIG9ubHkgYGVkZ2VfcHJlZGljdG9yX2Jlc3QucHRoYAooZmxhdCBtb2RlbCB3ZWlnaHRzKSDigJQgYSBzZXNzaW9uIHRoYXQgZGllcyBsb3NlcyBldmVyeSBlcG9jaCBzaW5jZSB0aGUgbGFzdApiZXN0LiBGb3IgQ29sYWIgUHJvJ3MgdW5pdC1zbGljZWQgcmVhbGl0eSAoc2Vzc2lvbnMgb2YgYSBmZXcgaG91cnMsIHNwcmVhZCBvdmVyCmRheXMpLCB0aGUgdHJhaW5lciBuZWVkcyBleGFjdGx5IHdoYXQgd2UgYnVpbHQgZm9yIGN0cmFjayBpbiByb3cgNDA6CgogICogYGNoZWNrcG9pbnRfbGFzdC5wdGhgIOKAlCBtb2RlbCArIG9wdGltaXplciArIGVwb2NoICsgYmVzdF9zY29yZSwgd3JpdHRlbgogICAgQVRPTUlDQUxMWSBldmVyeSBlcG9jaDsKICAqIGAtLXJlc3VtZSBhdXRvYCDigJQgY29udGludWUgZnJvbSBpdCBpZiBwcmVzZW50OwogICogYC0tbWF4LW1pbnV0ZXNgIOKAlCByZWZ1c2UgdG8gc3RhcnQgYW4gZXBvY2ggdGhhdCBjYW4ndCBmaW5pc2ggaW5zaWRlIHRoZQogICAgc2Vzc2lvbiBidWRnZXQsIHNvIHRoZSBzZXNzaW9uIGVuZHMgY2xlYW5seSB3aXRoIHRoZSBjaGVja3BvaW50IHVwbG9hZGFibGUuCgpBcHBsaWVkIHdpdGggZXhhY3QtY291bnQgYXNzZXJ0cyAoYSBwYXRjaCB0aGF0IHNpbGVudGx5IGZhaWxzIHRvIGFwcGx5IGxvb2tzCmV4YWN0bHkgbGlrZSBzdWNjZXNzKS4gU2hhcmVkIGJ5IHRoZSBsb2NhbCBNMSBzbW9rZSBhbmQgdGhlIENvbGFiIG5vdGVib29rIHNvCmJvdGggcnVuIHRoZSBzYW1lIGNvZGUuCgogICAgcHl0aG9uIHNjcmlwdHMvbG9jYWwvcGF0Y2hfZnVsbHJlY2lwZV90cmFpbmVyLnB5IDxwYXRoLXRvLXRyYWluX3VuZXRfdHJhbnNmb3JtZXIucHk+CiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCkFOQ0hPUl9BUkdTID0gJycnICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2luZ2xlLWdwdSIsIGRlc3Q9ImRhdGFfcGFyYWxsZWwiLCBhY3Rpb249InN0b3JlX2ZhbHNlIiwnJycKUEFUQ0hfQVJHUyA9ICcnJyAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJlc3VtZSIsIHR5cGU9c3RyLCBkZWZhdWx0PSJhdXRvIiwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iJ2F1dG8nIHJlc3VtZXMgZnJvbSBjaGVja3BvaW50X2xhc3QucHRoIGluIHRoZSBvdXRwdXQgZGlyIGlmICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAicHJlc2VudDsgJ25vbmUnIHN0YXJ0cyBmcmVzaCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1heC1taW51dGVzIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJzdG9wIGJlZm9yZSBzdGFydGluZyBhbiBlcG9jaCB0aGF0IHdvdWxkIGV4Y2VlZCB0aGlzIGJ1ZGdldCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXBhdGllbmNlIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTAsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9ImVhcmx5LXN0b3AgYWZ0ZXIgdGhpcyBtYW55IGVwb2NocyB3aXRob3V0IGEgbmV3IGJlc3QgKDAgPSBvZmYpIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2luZ2xlLWdwdSIsIGRlc3Q9ImRhdGFfcGFyYWxsZWwiLCBhY3Rpb249InN0b3JlX2ZhbHNlIiwnJycKCkFOQ0hPUl9TSUcgPSAnJycgICAgbWF4X2l0ZXJzOiBpbnQgfCBOb25lID0gTm9uZSwKICAgIGRlYnVnX3ZpZGVvOiBQYXRoIHwgTm9uZSA9IE5vbmUsJycnClBBVENIX1NJRyA9ICcnJyAgICBtYXhfaXRlcnM6IGludCB8IE5vbmUgPSBOb25lLAogICAgcmVzdW1lOiBzdHIgPSAiYXV0byIsCiAgICBtYXhfbWludXRlczogZmxvYXQgfCBOb25lID0gTm9uZSwKICAgIHBhdGllbmNlOiBpbnQgPSAxMCwKICAgIGRlYnVnX3ZpZGVvOiBQYXRoIHwgTm9uZSA9IE5vbmUsJycnCgpBTkNIT1JfQ0FMTCA9ICcnJyAgICAgICAgICAgIG1heF9pdGVycz1hcmdzLm1heF9pdGVycywnJycKUEFUQ0hfQ0FMTCA9ICcnJyAgICAgICAgICAgIG1heF9pdGVycz1hcmdzLm1heF9pdGVycywKICAgICAgICAgICAgcmVzdW1lPWFyZ3MucmVzdW1lLAogICAgICAgICAgICBtYXhfbWludXRlcz1hcmdzLm1heF9taW51dGVzLAogICAgICAgICAgICBwYXRpZW5jZT1hcmdzLnBhdGllbmNlLCcnJwoKQU5DSE9SX0xPT1AgPSAnJycgICAgYmVzdF9zY29yZSA9IDAuMAogICAgc2F2ZV9wYXRoID0gb3V0cHV0X2RpciAvICJlZGdlX3ByZWRpY3Rvcl9iZXN0LnB0aCIKICAgIHBiYXIgPSB0cWRtKHJhbmdlKG5fZXBvY2hzKSwgZGVzYz0iVHJhaW5pbmciLCBkaXNhYmxlPUZhbHNlKQogICAgcHJpbnQoZiJEZXRlY3Rpb24gbG9zczogd2VpZ2h0PXtkZXRfbG9zc193ZWlnaHR9LCBuZWdfd2VpZ2h0PXtkZXRfbmVnX3dlaWdodH0iLCBmbHVzaD1UcnVlKQoKICAgIGZvciBlcG9jaCBpbiBwYmFyOgogICAgICAgIHQwID0gdGltZS5tb25vdG9uaWMoKScnJwpQQVRDSF9MT09QID0gJycnICAgIGJlc3Rfc2NvcmUgPSAwLjAKICAgIHNhdmVfcGF0aCA9IG91dHB1dF9kaXIgLyAiZWRnZV9wcmVkaWN0b3JfYmVzdC5wdGgiCiAgICBsYXN0X3BhdGggPSBvdXRwdXRfZGlyIC8gImNoZWNrcG9pbnRfbGFzdC5wdGgiCgogICAgIyAtLS0tIHJlc3VtZSAodW5pdC1zbGljZWQgQ29sYWIgc2Vzc2lvbnM7IFNUUkFURUdZIDQ4KSAtLS0tCiAgICBzdGFydF9lcG9jaCA9IDAKICAgIGlmIHJlc3VtZSA9PSAiYXV0byIgYW5kIGxhc3RfcGF0aC5leGlzdHMoKToKICAgICAgICBfY2sgPSB0b3JjaC5sb2FkKGxhc3RfcGF0aCwgbWFwX2xvY2F0aW9uPWRldmljZSwgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgICAgIF9zdGF0ZSA9IF9ja1sibW9kZWxfc3RhdGVfZGljdCJdCiAgICAgICAgaWYgaXNpbnN0YW5jZShtb2RlbC51bmV0LCBubi5EYXRhUGFyYWxsZWwpOgogICAgICAgICAgICBfc3RhdGUgPSB7KGsucmVwbGFjZSgidW5ldC4iLCAidW5ldC5tb2R1bGUuIiwgMSkKICAgICAgICAgICAgICAgICAgICAgICBpZiBrLnN0YXJ0c3dpdGgoInVuZXQuIikgYW5kIG5vdCBrLnN0YXJ0c3dpdGgoInVuZXQubW9kdWxlLiIpIGVsc2Ugayk6IHYKICAgICAgICAgICAgICAgICAgICAgIGZvciBrLCB2IGluIF9zdGF0ZS5pdGVtcygpfQogICAgICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChfc3RhdGUpCiAgICAgICAgb3B0aW1pemVyLmxvYWRfc3RhdGVfZGljdChfY2tbIm9wdGltaXplcl9zdGF0ZV9kaWN0Il0pCiAgICAgICAgc3RhcnRfZXBvY2ggPSBpbnQoX2NrWyJlcG9jaCJdKSArIDEKICAgICAgICBiZXN0X3Njb3JlID0gZmxvYXQoX2NrLmdldCgiYmVzdF9zY29yZSIsIDAuMCkpCiAgICAgICAgc2luY2VfYmVzdCA9IGludChfY2suZ2V0KCJzaW5jZV9iZXN0IiwgMCkpCiAgICAgICAgcHJpbnQoZiJSRVNVTUVEIGZyb20ge2xhc3RfcGF0aC5uYW1lfTogZXBvY2gge3N0YXJ0X2Vwb2NofSAiCiAgICAgICAgICAgICAgZiIoYmVzdCB7YmVzdF9zY29yZTouNGZ9LCB7bGVuKF9ja1snbW9kZWxfc3RhdGVfZGljdCddKX0gdGVuc29ycykiLCBmbHVzaD1UcnVlKQogICAgZWxpZiByZXN1bWUgPT0gImF1dG8iOgogICAgICAgIHByaW50KCJmcmVzaCBydW4gKG5vIGNoZWNrcG9pbnRfbGFzdC5wdGgpIiwgZmx1c2g9VHJ1ZSkKCiAgICBzaW5jZV9iZXN0ID0gbG9jYWxzKCkuZ2V0KCJzaW5jZV9iZXN0IiwgMCkKICAgIF9ydW5fdDAgPSB0aW1lLm1vbm90b25pYygpCiAgICBfZXN0X2Vwb2NoX3MgPSBOb25lCiAgICBwYmFyID0gdHFkbShyYW5nZShzdGFydF9lcG9jaCwgbl9lcG9jaHMpLCBkZXNjPSJUcmFpbmluZyIsIGRpc2FibGU9RmFsc2UpCiAgICBwcmludChmIkRldGVjdGlvbiBsb3NzOiB3ZWlnaHQ9e2RldF9sb3NzX3dlaWdodH0sIG5lZ193ZWlnaHQ9e2RldF9uZWdfd2VpZ2h0fSIsIGZsdXNoPVRydWUpCgogICAgZm9yIGVwb2NoIGluIHBiYXI6CiAgICAgICAgaWYgbWF4X21pbnV0ZXMgaXMgbm90IE5vbmUgYW5kIF9lc3RfZXBvY2hfcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgX2VsYXBzZWRfbWluID0gKHRpbWUubW9ub3RvbmljKCkgLSBfcnVuX3QwKSAvIDYwLjAKICAgICAgICAgICAgaWYgX2VsYXBzZWRfbWluICsgX2VzdF9lcG9jaF9zIC8gNjAuMCA+IG1heF9taW51dGVzOgogICAgICAgICAgICAgICAgcHJpbnQoZiJCVURHRVQgU1RPUDoge19lbGFwc2VkX21pbjouMWZ9IG1pbiBlbGFwc2VkLCBuZXh0IGVwb2NoICIKICAgICAgICAgICAgICAgICAgICAgIGYifntfZXN0X2Vwb2NoX3MvNjA6LjFmfSBtaW4gZXhjZWVkcyB7bWF4X21pbnV0ZXM6LjBmfSBtaW4uICIKICAgICAgICAgICAgICAgICAgICAgIGYiUmVzdW1hYmxlIGF0IGVwb2NoIHtlcG9jaH0uIiwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgdDAgPSB0aW1lLm1vbm90b25pYygpJycnCgpBTkNIT1JfU0FWRSA9ICcnJyAgICAgICAgbWFya2VyID0gIioiIGlmIGlzX2Jlc3QgZWxzZSAiICInJycKUEFUQ0hfU0FWRSA9ICcnJyAgICAgICAgIyAtLS0tIGF0b21pYyByZXN1bWFibGUgY2hlY2twb2ludCwgZXZlcnkgZXBvY2ggLS0tLQogICAgICAgIF9lc3RfZXBvY2hfcyA9IHRyYWluX3RpbWUgKyB0ZXN0X3RpbWUKICAgICAgICBfdG1wID0gbGFzdF9wYXRoLndpdGhfc3VmZml4KCIudG1wIikKICAgICAgICBzaW5jZV9iZXN0ID0gMCBpZiBpc19iZXN0IGVsc2Ugc2luY2VfYmVzdCArIDEKICAgICAgICB0b3JjaC5zYXZlKHsKICAgICAgICAgICAgImVwb2NoIjogZXBvY2gsCiAgICAgICAgICAgICJiZXN0X3Njb3JlIjogYmVzdF9zY29yZSwKICAgICAgICAgICAgInNpbmNlX2Jlc3QiOiBzaW5jZV9iZXN0LAogICAgICAgICAgICAibW9kZWxfc3RhdGVfZGljdCI6IHtrLnJlcGxhY2UoInVuZXQubW9kdWxlLiIsICJ1bmV0LiIsIDEpOiB2CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBrLCB2IGluIG1vZGVsLnN0YXRlX2RpY3QoKS5pdGVtcygpfSwKICAgICAgICAgICAgIm9wdGltaXplcl9zdGF0ZV9kaWN0Ijogb3B0aW1pemVyLnN0YXRlX2RpY3QoKSwKICAgICAgICB9LCBfdG1wKQogICAgICAgIF90bXAucmVwbGFjZShsYXN0X3BhdGgpCgogICAgICAgIGlmIHBhdGllbmNlIGFuZCBzaW5jZV9iZXN0ID49IHBhdGllbmNlOgogICAgICAgICAgICBwcmludChmIkVBUkxZIFNUT1A6IG5vIG5ldyBiZXN0IGluIHtwYXRpZW5jZX0gZXBvY2hzICIKICAgICAgICAgICAgICAgICAgZiIoYmVzdCB7YmVzdF9zY29yZTouNGZ9KS4gU3RvcHBpbmcgYXQgZXBvY2gge2Vwb2NofS4iLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBfcHJpbnRfZmluYWwgPSBUcnVlCiAgICAgICAgICAgIG1hcmtlciA9ICIqIiBpZiBpc19iZXN0IGVsc2UgIiAiCiAgICAgICAgICAgIHByaW50KGYiICBFcG9jaCB7ZXBvY2g6M2R9L3tuX2Vwb2Noc30gfCAoZWFybHktc3RvcCBlcG9jaCkiLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBicmVhawoKICAgICAgICBtYXJrZXIgPSAiKiIgaWYgaXNfYmVzdCBlbHNlICIgIicnJwoKCmRlZiBwYXRjaChwYXRoOiBQYXRoKSAtPiBOb25lOgogICAgcyA9IHBhdGgucmVhZF90ZXh0KCkKICAgIGlmICJCVURHRVQgU1RPUCIgaW4gczoKICAgICAgICBwcmludCgiYWxyZWFkeSBwYXRjaGVkIikKICAgICAgICByZXR1cm4KICAgIGZvciBuYW1lLCBvbGQsIG5ldyBpbiBbKCJhcmdzIiwgQU5DSE9SX0FSR1MsIFBBVENIX0FSR1MpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAoInNpZyIsIEFOQ0hPUl9TSUcsIFBBVENIX1NJRyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICgiY2FsbCIsIEFOQ0hPUl9DQUxMLCBQQVRDSF9DQUxMKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgKCJsb29wIiwgQU5DSE9SX0xPT1AsIFBBVENIX0xPT1ApLAogICAgICAgICAgICAgICAgICAgICAgICAgICAoInNhdmUiLCBBTkNIT1JfU0FWRSwgUEFUQ0hfU0FWRSldOgogICAgICAgIG4gPSBzLmNvdW50KG9sZCkKICAgICAgICBhc3NlcnQgbiA9PSAxLCBmImFuY2hvciB7bmFtZSFyfSBtYXRjaGVkIHtufSB0aW1lcywgZXhwZWN0ZWQgMSIKICAgICAgICBzID0gcy5yZXBsYWNlKG9sZCwgbmV3LCAxKQogICAgY29tcGlsZShzLCBzdHIocGF0aCksICJleGVjIikKICAgIHBhdGgud3JpdGVfdGV4dChzKQogICAgcHJpbnQoZiJwYXRjaGVkIHtwYXRofSAoNS81IGFuY2hvcnMsIGNvbXBpbGUgT0spIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcGF0Y2goUGF0aChzeXMuYXJndlsxXSkpCg=="))
!cd /content && python patcher.py /content/repo/scripts/train_unet_transformer.py

pathlib.Path('/content/w3_probe_splits.json').write_bytes(base64.b64decode("WwogewogICJ0cmFpbiI6IFsKICAgIjQ0YjZfMGRiNzVmYWUuemFyciIsCiAgICI0NGI2XzEyZGZiMzkxLnphcnIiLAogICAiNDRiNl8xNTc0ODAyYi56YXJyIiwKICAgIjQ0YjZfMThjZWQ4MTguemFyciIsCiAgICI0NGI2XzFkNTMwODMxLnphcnIiLAogICAiNDRiNl8yNDI2NGYxMi56YXJyIiwKICAgIjQ0YjZfMjY3MTQ4ZTQuemFyciIsCiAgICI0NGI2XzJhMmVmZjlmLnphcnIiLAogICAiNDRiNl8zNDFkZjI1Zi56YXJyIiwKICAgIjQ0YjZfM2JiMzY5MGYuemFyciIsCiAgICI0NGI2XzUzZjk1MjUyLnphcnIiLAogICAiNDRiNl81NTFhNWRiYS56YXJyIiwKICAgIjQ0YjZfNWYxNWQxMzUuemFyciIsCiAgICI0NGI2XzY2ZjkyOTJkLnphcnIiLAogICAiNDRiNl83MDYwOTJmMC56YXJyIiwKICAgIjQ0YjZfNzFhNDE3OWYuemFyciIsCiAgICI0NGI2XzdhMzAyZGEwLnphcnIiLAogICAiNDRiNl83ZTU1NzcwOS56YXJyIiwKICAgIjQ0YjZfODA4OTUyZDYuemFyciIsCiAgICI0NGI2XzkwNzI0ODkyLnphcnIiLAogICAiNDRiNl85NTAyOWU5Mi56YXJyIiwKICAgIjQ0YjZfOWJlODBiMDQuemFyciIsCiAgICI0NGI2X2EyMTEyMGMyLnphcnIiLAogICAiNDRiNl9iMmM0NDI2Ni56YXJyIiwKICAgIjQ0YjZfYzc3MWNiMDQuemFyciIsCiAgICI0NGI2X2NmMjUzNmU4LnphcnIiLAogICAiNDRiNl9kMjljOWFiMi56YXJyIiwKICAgIjQ0YjZfZDJmMzRmOTAuemFyciIsCiAgICI0NGI2X2RiM2M4NDdiLnphcnIiLAogICAiNDRiNl9kZGY1NzdhZC56YXJyIiwKICAgIjQ0YjZfZTI4ODQwYzYuemFyciIsCiAgICI0NGI2X2UyOWYwMTc2LnphcnIiLAogICAiNDRiNl9mMjg3MDdjNi56YXJyIiwKICAgIjZiYmFfMGM3ZmE3MTguemFyciIsCiAgICI2YmJhXzEzZjE5NTMxLnphcnIiLAogICAiNmJiYV8xZDBkODM4NC56YXJyIiwKICAgIjZiYmFfMWViZmI4MGQuemFyciIsCiAgICI2YmJhXzFmNThjMmY2LnphcnIiLAogICAiNmJiYV8yMDg1MjgxOC56YXJyIiwKICAgIjZiYmFfMjMxMmFjNDEuemFyciIsCiAgICI2YmJhXzIzYWY5ZWViLnphcnIiLAogICAiNmJiYV8yNjhlMTIzMC56YXJyIiwKICAgIjZiYmFfMzEyZjBkYzMuemFyciIsCiAgICI2YmJhXzMyZGIxM2ZjLnphcnIiLAogICAiNmJiYV8zNzJjOGNiOC56YXJyIiwKICAgIjZiYmFfM2FiZmUxMGEuemFyciIsCiAgICI2YmJhXzNjNTY5MWI2LnphcnIiLAogICAiNmJiYV8zZmRhNmIyNS56YXJyIiwKICAgIjZiYmFfNDNmZWEzOWQuemFyciIsCiAgICI2YmJhXzRhYTE0Yzg5LnphcnIiLAogICAiNmJiYV81YTRkOTM2MC56YXJyIiwKICAgIjZiYmFfNWIyODQ3MmEuemFyciIsCiAgICI2YmJhXzVkZmU5YWQxLnphcnIiLAogICAiNmJiYV82N2ViZDA3My56YXJyIiwKICAgIjZiYmFfNmNhODczNzAuemFyciIsCiAgICI2YmJhXzZkYjBlOWI0LnphcnIiLAogICAiNmJiYV82ZmViMTBmMC56YXJyIiwKICAgIjZiYmFfNmZlZWIwYjEuemFyciIsCiAgICI2YmJhXzcwNWVjMmM5LnphcnIiLAogICAiNmJiYV83NDY4NmQ2YS56YXJyIiwKICAgIjZiYmFfNzY3YTFlMTcuemFyciIsCiAgICI2YmJhXzc2ZGI3OGMxLnphcnIiLAogICAiNmJiYV83OGE3YmQ5Ny56YXJyIiwKICAgIjZiYmFfN2Y4N2IzZDguemFyciIsCiAgICI2YmJhXzhiNzgxOGJmLnphcnIiLAogICAiNmJiYV85MDcyNzFkYi56YXJyIiwKICAgIjZiYmFfOTE5NTFiM2EuemFyciIsCiAgICI2YmJhXzlhNDFkMDI5LnphcnIiLAogICAiNmJiYV85ZTIzNDMwYi56YXJyIiwKICAgIjZiYmFfYTkwYTBiOWMuemFyciIsCiAgICI2YmJhX2FjZDc4MmE4LnphcnIiLAogICAiNmJiYV9hZWVlNzgwNS56YXJyIiwKICAgIjZiYmFfYWYxNDljOTQuemFyciIsCiAgICI2YmJhX2FmYjE0MWZmLnphcnIiLAogICAiNmJiYV9iMWFlMzdiOS56YXJyIiwKICAgIjZiYmFfYjY5MzM4MWIuemFyciIsCiAgICI2YmJhX2JiOWYyMGMzLnphcnIiLAogICAiNmJiYV9jMjdjYmEwOC56YXJyIiwKICAgIjZiYmFfYzczYTFkMTEuemFyciIsCiAgICI2YmJhX2NkY2ZlNTMzLnphcnIiLAogICAiNmJiYV9jZjM1MjE0Yy56YXJyIiwKICAgIjZiYmFfY2ZmNTg2NWYuemFyciIsCiAgICI2YmJhX2QwZmMzOGI1LnphcnIiLAogICAiNmJiYV9kMWFjYjZmZi56YXJyIiwKICAgIjZiYmFfZDJiOWZjMGMuemFyciIsCiAgICI2YmJhX2QzZGE3NTNiLnphcnIiLAogICAiNmJiYV9kNWVhZTE3NS56YXJyIiwKICAgIjZiYmFfZTVlNDQ5ODguemFyciIsCiAgICI2YmJhX2ViZGYzYjM0LnphcnIiLAogICAiNmJiYV9lYmZmNmU3Ni56YXJyIiwKICAgIjZiYmFfZjE3YmVmYmMuemFyciIsCiAgICI2YmJhX2YxZmRlN2UwLnphcnIiLAogICAiNmJiYV9mMjA0NzhlOS56YXJyIiwKICAgIjZiYmFfZjRhZTgxMWMuemFyciIsCiAgICI2YmJhX2Y4ZmZkNWU3LnphcnIiLAogICAiNmJiYV9mYzUxNmRjNi56YXJyIgogIF0sCiAgInRlc3QiOiBbCiAgICI0NGI2XzU3NDBkMjRiLnphcnIiLAogICAiNDRiNl84Y2M2NTA2Yy56YXJyIiwKICAgIjQ0YjZfOGY5ZWNhYjQuemFyciIsCiAgICI2YmJhXzA2MmM4ZDM3LnphcnIiLAogICAiNmJiYV8yMDdjNmFhZi56YXJyIiwKICAgIjZiYmFfMjU0MGNkOTAuemFyciIsCiAgICI2YmJhXzNhMTg0OWMyLnphcnIiLAogICAiNmJiYV83ODRhNzhjOS56YXJyIiwKICAgIjZiYmFfOTcxZmE1ZTAuemFyciIsCiAgICI2YmJhX2E1ZTkyNmJiLnphcnIiLAogICAiNmJiYV9hYjc4NDEzZC56YXJyIiwKICAgIjZiYmFfZmM4MzgzN2QuemFyciIKICBdLAogICJfdHJhaW5fZnVsbCI6IFsKICAgIjQ0YjZfMDExM2RlM2IuemFyciIsCiAgICI0NGI2XzBiMjQ4NDVmLnphcnIiLAogICAiNDRiNl8wYzU4MmZkYy56YXJyIiwKICAgIjQ0YjZfMGRiNzVmYWUuemFyciIsCiAgICI0NGI2XzEyZGZiMzkxLnphcnIiLAogICAiNDRiNl8xNDRiMjU2ZC56YXJyIiwKICAgIjQ0YjZfMTU3NDgwMmIuemFyciIsCiAgICI0NGI2XzE4Y2VkODE4LnphcnIiLAogICAiNDRiNl8xZDUzMDgzMS56YXJyIiwKICAgIjQ0YjZfMjQyNjRmMTIuemFyciIsCiAgICI0NGI2XzI2NzE0OGU0LnphcnIiLAogICAiNDRiNl8yYTJlZmY5Zi56YXJyIiwKICAgIjQ0YjZfMzNiNTk2YmYuemFyciIsCiAgICI0NGI2XzM0MWRmMjVmLnphcnIiLAogICAiNDRiNl8zYTg2MWUwMy56YXJyIiwKICAgIjQ0YjZfM2JiMzY5MGYuemFyciIsCiAgICI0NGI2XzQwYzQ1ZjVhLnphcnIiLAogICAiNDRiNl80MTVjMGEzYS56YXJyIiwKICAgIjQ0YjZfNTNmOTUyNTIuemFyciIsCiAgICI0NGI2XzU1MWE1ZGJhLnphcnIiLAogICAiNDRiNl81ODdhMWUyMi56YXJyIiwKICAgIjQ0YjZfNWYxNWQxMzUuemFyciIsCiAgICI0NGI2XzY2OGUwY2M3LnphcnIiLAogICAiNDRiNl82NmY5MjkyZC56YXJyIiwKICAgIjQ0YjZfNzA2MDkyZjAuemFyciIsCiAgICI0NGI2XzcxYTQxNzlmLnphcnIiLAogICAiNDRiNl83NGQwYzUyZS56YXJyIiwKICAgIjQ0YjZfN2EzMDJkYTAuemFyciIsCiAgICI0NGI2XzdlNTU3NzA5LnphcnIiLAogICAiNDRiNl84MDg5NTJkNi56YXJyIiwKICAgIjQ0YjZfODFjMjU2ZjAuemFyciIsCiAgICI0NGI2Xzg3YmJhNmM0LnphcnIiLAogICAiNDRiNl84ZjVhYjkzMS56YXJyIiwKICAgIjQ0YjZfOTA3MjQ4OTIuemFyciIsCiAgICI0NGI2Xzk0OWFkZWIxLnphcnIiLAogICAiNDRiNl85NTAyOWU5Mi56YXJyIiwKICAgIjQ0YjZfOTc3MjUxNDQuemFyciIsCiAgICI0NGI2Xzk5NjE1NWRlLnphcnIiLAogICAiNDRiNl85YmU4MGIwNC56YXJyIiwKICAgIjQ0YjZfOWJmYTZhMGEuemFyciIsCiAgICI0NGI2X2EyMTEyMGMyLnphcnIiLAogICAiNDRiNl9hYWY4YjBlYS56YXJyIiwKICAgIjQ0YjZfYWJmODI1MTguemFyciIsCiAgICI0NGI2X2IyYzQ0MjY2LnphcnIiLAogICAiNDRiNl9jMTVmZGVkMi56YXJyIiwKICAgIjQ0YjZfYzUwMjA0ZTAuemFyciIsCiAgICI0NGI2X2M3NzFjYjA0LnphcnIiLAogICAiNDRiNl9jOGUyYTUyMy56YXJyIiwKICAgIjQ0YjZfY2YyNTM2ZTguemFyciIsCiAgICI0NGI2X2NmOGZlZDZiLnphcnIiLAogICAiNDRiNl9kMjljOWFiMi56YXJyIiwKICAgIjQ0YjZfZDJmMzRmOTAuemFyciIsCiAgICI0NGI2X2Q1ZTdkODkxLnphcnIiLAogICAiNDRiNl9kNzU0YWE1OS56YXJyIiwKICAgIjQ0YjZfZDc4ZTA5ZDkuemFyciIsCiAgICI0NGI2X2RiM2M4NDdiLnphcnIiLAogICAiNDRiNl9kZGY1NzdhZC56YXJyIiwKICAgIjQ0YjZfZGVhYmFjOTUuemFyciIsCiAgICI0NGI2X2UyODg0MGM2LnphcnIiLAogICAiNDRiNl9lMjlmMDE3Ni56YXJyIiwKICAgIjQ0YjZfZTMxMjYxYjQuemFyciIsCiAgICI0NGI2X2UzNWIxMTdkLnphcnIiLAogICAiNDRiNl9lNTdmZjVjNi56YXJyIiwKICAgIjQ0YjZfZjI4NzA3YzYuemFyciIsCiAgICI2YmJhXzA1YjY4NTBiLnphcnIiLAogICAiNmJiYV8wNWRiMGZiMS56YXJyIiwKICAgIjZiYmFfMDc0NzcwMzMuemFyciIsCiAgICI2YmJhXzA3ZTI0MTMyLnphcnIiLAogICAiNmJiYV8wODViZjY1Ni56YXJyIiwKICAgIjZiYmFfMDk5NjEyOTIuemFyciIsCiAgICI2YmJhXzBjN2ZhNzE4LnphcnIiLAogICAiNmJiYV8wZTdjMGQwNy56YXJyIiwKICAgIjZiYmFfMTI2NjVjMGUuemFyciIsCiAgICI2YmJhXzEzZjE5NTMxLnphcnIiLAogICAiNmJiYV8xNTQwM2I5YS56YXJyIiwKICAgIjZiYmFfMWQwZDgzODQuemFyciIsCiAgICI2YmJhXzFlYmZiODBkLnphcnIiLAogICAiNmJiYV8xZjU4YzJmNi56YXJyIiwKICAgIjZiYmFfMjA4NTI4MTguemFyciIsCiAgICI2YmJhXzIzMTJhYzQxLnphcnIiLAogICAiNmJiYV8yM2FmOWVlYi56YXJyIiwKICAgIjZiYmFfMjY0NmFmYzcuemFyciIsCiAgICI2YmJhXzI2OGUxMjMwLnphcnIiLAogICAiNmJiYV8yODE5Y2ExNC56YXJyIiwKICAgIjZiYmFfMjgzYmY5ZjEuemFyciIsCiAgICI2YmJhXzMxMmYwZGMzLnphcnIiLAogICAiNmJiYV8zMmRiMTNmYy56YXJyIiwKICAgIjZiYmFfMzM3YjFiM2EuemFyciIsCiAgICI2YmJhXzM3MmM4Y2I4LnphcnIiLAogICAiNmJiYV8zYWJmZTEwYS56YXJyIiwKICAgIjZiYmFfM2M1NjkxYjYuemFyciIsCiAgICI2YmJhXzNkYjU0ZTIwLnphcnIiLAogICAiNmJiYV8zZmRhNmIyNS56YXJyIiwKICAgIjZiYmFfNDNmZWEzOWQuemFyciIsCiAgICI2YmJhXzQ3NGJlNjY0LnphcnIiLAogICAiNmJiYV80ODgxNjEyMS56YXJyIiwKICAgIjZiYmFfNGFhMTRjODkuemFyciIsCiAgICI2YmJhXzRmOTljZTIwLnphcnIiLAogICAiNmJiYV80ZmZkM2RhMy56YXJyIiwKICAgIjZiYmFfNTViN2VlYmUuemFyciIsCiAgICI2YmJhXzU1YzcwODQzLnphcnIiLAogICAiNmJiYV81N2I3Y2MxZS56YXJyIiwKICAgIjZiYmFfNWE0ZDkzNjAuemFyciIsCiAgICI2YmJhXzViMjg0NzJhLnphcnIiLAogICAiNmJiYV81YzAzOTg5NS56YXJyIiwKICAgIjZiYmFfNWM4MjQ4NzYuemFyciIsCiAgICI2YmJhXzVkZmU5YWQxLnphcnIiLAogICAiNmJiYV81Zjg5MDM5ZC56YXJyIiwKICAgIjZiYmFfNjFkZDFlMGQuemFyciIsCiAgICI2YmJhXzYxZWNiZTY1LnphcnIiLAogICAiNmJiYV82MzIxYTM1OS56YXJyIiwKICAgIjZiYmFfNjQ3OTQzNWQuemFyciIsCiAgICI2YmJhXzY3ZWJkMDczLnphcnIiLAogICAiNmJiYV82Y2E4NzM3MC56YXJyIiwKICAgIjZiYmFfNmRiMGU5YjQuemFyciIsCiAgICI2YmJhXzZmZWIxMGYwLnphcnIiLAogICAiNmJiYV82ZmVlYjBiMS56YXJyIiwKICAgIjZiYmFfNzA1ZWMyYzkuemFyciIsCiAgICI2YmJhXzcxOGIyMWY5LnphcnIiLAogICAiNmJiYV83NDY4NmQ2YS56YXJyIiwKICAgIjZiYmFfNzY3YTFlMTcuemFyciIsCiAgICI2YmJhXzc2ZGI3OGMxLnphcnIiLAogICAiNmJiYV83ODY4OTNhYy56YXJyIiwKICAgIjZiYmFfNzg5ZjgxNjguemFyciIsCiAgICI2YmJhXzc4YTdiZDk3LnphcnIiLAogICAiNmJiYV83YWY1NGZkZS56YXJyIiwKICAgIjZiYmFfN2QzMDU4YWUuemFyciIsCiAgICI2YmJhXzdmODdiM2Q4LnphcnIiLAogICAiNmJiYV84MGQxMjgyNC56YXJyIiwKICAgIjZiYmFfODI1YmQxYzYuemFyciIsCiAgICI2YmJhXzg3Mjg5ZTEzLnphcnIiLAogICAiNmJiYV84Yjc4MThiZi56YXJyIiwKICAgIjZiYmFfOTA3MjcxZGIuemFyciIsCiAgICI2YmJhXzkxOTUxYjNhLnphcnIiLAogICAiNmJiYV85NjgzMzM4NC56YXJyIiwKICAgIjZiYmFfOTY5NjE4ZjYuemFyciIsCiAgICI2YmJhXzlhNDFkMDI5LnphcnIiLAogICAiNmJiYV85ZTIzNDMwYi56YXJyIiwKICAgIjZiYmFfYTkwYTBiOWMuemFyciIsCiAgICI2YmJhX2FjZDc4MmE4LnphcnIiLAogICAiNmJiYV9hZTgyYTc5MS56YXJyIiwKICAgIjZiYmFfYWVlZTc4MDUuemFyciIsCiAgICI2YmJhX2FmMTQ5Yzk0LnphcnIiLAogICAiNmJiYV9hZmIxNDFmZi56YXJyIiwKICAgIjZiYmFfYjFhZTM3YjkuemFyciIsCiAgICI2YmJhX2IyMDRjYWM3LnphcnIiLAogICAiNmJiYV9iNjkzMzgxYi56YXJyIiwKICAgIjZiYmFfYmI5ZjIwYzMuemFyciIsCiAgICI2YmJhX2JiYjcwOGNhLnphcnIiLAogICAiNmJiYV9jMjdjYmEwOC56YXJyIiwKICAgIjZiYmFfYzMyOGYyZmQuemFyciIsCiAgICI2YmJhX2M3M2ExZDExLnphcnIiLAogICAiNmJiYV9jZGNmZTUzMy56YXJyIiwKICAgIjZiYmFfY2YzNTIxNGMuemFyciIsCiAgICI2YmJhX2NmZjU4NjVmLnphcnIiLAogICAiNmJiYV9kMGZjMzhiNS56YXJyIiwKICAgIjZiYmFfZDFhY2I2ZmYuemFyciIsCiAgICI2YmJhX2QyYjlmYzBjLnphcnIiLAogICAiNmJiYV9kM2RhNzUzYi56YXJyIiwKICAgIjZiYmFfZDVlYWUxNzUuemFyciIsCiAgICI2YmJhX2Q2ZWNlYmJiLnphcnIiLAogICAiNmJiYV9kZjY3M2E4My56YXJyIiwKICAgIjZiYmFfZTE2ZmZjNTguemFyciIsCiAgICI2YmJhX2U1ZTQ0OTg4LnphcnIiLAogICAiNmJiYV9lYmRmM2IzNC56YXJyIiwKICAgIjZiYmFfZWJmZjZlNzYuemFyciIsCiAgICI2YmJhX2VkOTM3N2ZkLnphcnIiLAogICAiNmJiYV9lZGYxNDU4My56YXJyIiwKICAgIjZiYmFfZWViYzU3YTUuemFyciIsCiAgICI2YmJhX2VmN2I0ZjdlLnphcnIiLAogICAiNmJiYV9mMTdiZWZiYy56YXJyIiwKICAgIjZiYmFfZjFmZGU3ZTAuemFyciIsCiAgICI2YmJhX2YyMDQ3OGU5LnphcnIiLAogICAiNmJiYV9mNGFlODExYy56YXJyIiwKICAgIjZiYmFfZjhmZmQ1ZTcuemFyciIsCiAgICI2YmJhX2ZiYzg5OGRjLnphcnIiLAogICAiNmJiYV9mYzUxNmRjNi56YXJyIiwKICAgIjZiYmFfZmM1ZjM5ZGMuemFyciIsCiAgICI2YmJhX2ZlNjcwMzIwLnphcnIiCiAgXSwKICAiX2dvbGRlbl9ob2xkb3V0X25ldmVyX3VzZWQiOiBbCiAgICI0NGI2XzJmMzFmYzJmLnphcnIiLAogICAiNDRiNl9hMmJiNDhiYi56YXJyIiwKICAgIjQ0YjZfYzk2Y2ZhMTAuemFyciIsCiAgICI0NGI2X2ViMjg4MGZjLnphcnIiLAogICAiNmJiYV83YjVkM2IyYy56YXJyIiwKICAgIjZiYmFfYjMyOWFmNDQuemFyciIsCiAgICI2YmJhX2Q4MmE0ZmM2LnphcnIiLAogICAiNmJiYV9kZWJkN2JmYS56YXJyIgogIF0KIH0KXQ=="))
sp = json.loads(pathlib.Path('/content/w3_probe_splits.json').read_text())[0]
print(f"splits: train {len(sp['train'])} | selection {len(sp['test'])} | golden quarantined {len(sp['_golden_holdout_never_used'])}")


In [ ]:
#@title Resume from the checkpoint store (if any)
import os, glob, shutil
out_dir = '/content/repo/weights/w3probe/split_0'
os.makedirs(out_dir, exist_ok=True)
!kaggle datasets download maruanbottoni/biohub-w3-ckpt -p /content/ckpt --unzip || true
ck = glob.glob('/content/ckpt/**/checkpoint_last.pth', recursive=True)
if ck:
    shutil.copy(ck[0], f'{out_dir}/checkpoint_last.pth')
    print('RESUME SEED:', ck[0])
    for extra in ('edge_predictor_best.pth', 'config.json'):
        e = glob.glob(f'/content/ckpt/**/{extra}', recursive=True)
        if e: shutil.copy(e[0], f'{out_dir}/{extra}')
else:
    print('fresh run — no checkpoint in the store yet')


In [ ]:
#@title TRAIN (window-3, resumable, budget-stopped)
import subprocess, sys, os
env = {**os.environ, 'PYTHONPATH': 'src'}
cmd = [sys.executable, 'scripts/train_unet_transformer.py',
       '--data-dir', '/content/data/train',
       '--splits', '/content/w3_probe_splits.json', '--split', '0',
       '--method', 'w3probe', '--window-size', '3',
       '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
       '--num-workers', '4', '--single-gpu',
       '--max-minutes', str(MAX_MINUTES),
       '--patience', '10']
print(' '.join(cmd))
r = subprocess.run(cmd, cwd='/content/repo', env=env)
print('trainer exit:', r.returncode)


In [ ]:
#@title Upload checkpoint back to the store (ALWAYS run this cell)
import json, os, shutil, pathlib, glob
st = pathlib.Path('/content/w3ckpt_staging'); st.mkdir(exist_ok=True)
src = '/content/repo/weights/w3probe/split_0'
n = 0
for f in ('checkpoint_last.pth', 'edge_predictor_best.pth', 'config.json'):
    if os.path.exists(f'{src}/{f}'):
        shutil.copy(f'{src}/{f}', st / f); n += 1
assert n >= 1, 'nothing to upload — did training run?'
(st / 'dataset-metadata.json').write_text(json.dumps({
    'id': 'maruanbottoni/biohub-w3-ckpt', 'title': 'biohub-w3-ckpt',
    'licenses': [{'name': 'CC0-1.0'}]}))
import torch
ck = torch.load(st / 'checkpoint_last.pth', map_location='cpu', weights_only=False)
msg = f"epoch {ck['epoch']} best {ck.get('best_score', 0):.4f} ({os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()})"
!kaggle datasets version -p /content/w3ckpt_staging -m "{msg}"
print('uploaded:', msg)
